### MessageState | entrada e saida de Schema | Estado privado

In [3]:
from dotenv import load_dotenv
import os
from langchain_openai import ChatOpenAI

#### Carregando API KEY e defininido modelo

In [4]:
load_dotenv()
API_KEY = os.getenv('OPENAI_KEY')
llm = ChatOpenAI(model='gpt-6-luna',api_key = API_KEY)

#### `MessagesState`

`MessagesState` é só um `TypedDict` com uma chave:

```python
class MessagesState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]
```

In [5]:
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import START,END,MessagesState,StateGraph

def responder(state: MessagesState) -> dict:
    return {'messages' : [llm.invoke(state['messages'])]}

#### Grafo

In [7]:
graph = StateGraph(MessagesState)
#node
graph.add_node('responder',responder)
#edges
graph.add_edge(START,'responder')
graph.add_edge('responder',END)
# configuração de memoria
chat = graph.compile(checkpointer=InMemorySaver())

#### Chamada usando checkpointer

In [13]:
# config do checkpointer
config = {'configurable':{'thread_id':'conversa-3'}}
chat.invoke({'messages': [("user", "Oi! Meu nome é Ana e eu trabalho com logística.")]},config)
resposta = chat.invoke({'messages': [("user", "Qual meu nome é area?")]},config)

for msg in resposta['messages']:
    print(f"{msg.type:>5}: {msg.content[:100]}")



human: Oi! Meu nome é Ana e eu trabalho com logística.
   ai: Oi, Ana! Prazer em conhecer você. Como posso ajudar com logística hoje?
human: Qual meu nome é area?
   ai: Seu nome é Ana e sua área de trabalho é logística.


> Na segunda chamada enviamos **só a nova mensagem**: o `add_messages` a acrescentou ao histórico salvo pelo checkpointer.

#### Estendendo o `MessagesState`

Aqui, um chatbot que conta as interações e detecta o idioma do usuário.

In [14]:
import operator
from typing import Annotated

In [15]:
class EstadoChat(MessagesState):
    idioma : str
    interacoes : Annotated[int,operator.add]

In [ ]:
def detectarIdioma(state: EstadoChat) -> dict:
    ultima = state['messages'][-1].content.lower()
    idioma = "en" if any(p in ultima.split() for p in ("the", "is", "what", "hello", "my")) else "pt"
    return {'idioma' : idioma, 'interacoes' : 1}

def responderNoIdioma(state: EstadoChat) -> dict:
    system_prompt = 'Responda em portugues, em uma frase' if state['idioma'] == 'pt' else 'Answer in English, in one sentence.'
    return {'messages': [llm.invoke([('system',system_prompt)] + state['messages'])]}

In [27]:
grafo = StateGraph(EstadoChat)
grafo.add_node('detectar',detectarIdioma)
grafo.add_node('responder',responderNoIdioma)
grafo.add_edge(START,'detectar')
grafo.add_edge('detectar','responder')
grafo.add_edge('responder',END)

chat2 = grafo.compile(checkpointer=InMemorySaver())

In [31]:
cfg = {'configurable' : {'thread_id' : 'conversa_04'}}
for pergunta in ["Qual a capital do Brasil?", "What is the capital of France?"]:
    r = chat2.invoke({"messages": [("user", pergunta)]}, cfg)
    print(f"[{r['idioma']}] interações={r['interacoes']} → {r['messages'][-1].content[:90]}")

[pt] interações=5 → A capital do Brasil é Brasília.
[pt] interações=6 → A capital da França é Paris.
